In [2]:
import pandas as pd
import numpy as np

# ==========================
# LOAD DATA
# ==========================

file="../ADIDatabase_2015.csv"

df=pd.read_csv(
    file,
    dtype=str
)

print("Original Shape:",df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

empty_values=[

    "",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "nan"

]

df=df.replace(
    empty_values,
    np.nan
)

# ==========================
# HANDLE SERIAL COLUMN
# UNNAMED -> SI_No
# ==========================

si_col=None

for col in df.columns:

    c=(
        str(col)
        .lower()
        .replace("_","")
        .replace(" ","")
    )

    if "unnamed" in c:

        si_col=col
        break

if si_col:

    df=df.rename(
        columns={
            si_col:"SI_No"
        }
    )

# ==========================
# REMOVE UNKNOWN COLUMNS
# KEEP SI_No
# ==========================

unknown=[]

for col in df.columns:

    if (
        "unnamed" in col.lower()
        and col!="SI_No"
    ):

        unknown.append(col)

df=df.drop(
    columns=unknown,
    errors="ignore"
)

print("\nUnknown Columns Removed:")
print(unknown)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    empty=(

        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()

    )

    if empty:

        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:",duplicates)

# ==========================
# RESET SI_No
# ==========================

df=df.reset_index(
    drop=True
)

if "SI_No" in df.columns:

    df["SI_No"]=range(
        1,
        len(df)+1
    )

else:

    df.insert(
        0,
        "SI_No",
        range(
            1,
            len(df)+1
        )
    )

print("\nSI_No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(

    df
    .replace("",np.nan)
    .isna()
    .sum()

)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns After Cleaning:")
print(list(df.columns))

# ==========================
# REMOVE ILLEGAL EXCEL CHARS
# ==========================

import re

def clean_excel_text(v):

    if pd.isna(v):
        return v

    v=str(v)

    v=re.sub(
        r"[\x00-\x08\x0B\x0C\x0E-\x1F]",
        "",
        v
    )

    return v

df=df.apply(
    lambda col:
    col.map(clean_excel_text)
)

# ==========================
# EXPORT
# ==========================

output="cleaned_ADIDatabase_2015.xlsx"

with pd.ExcelWriter(
    output,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        index=False,
        sheet_name="Cleaned_Data"
    )

print("\nSaved:",output)

Original Shape: (1411, 36)

Unknown Columns Removed:
[]

Removed Empty Columns:
['IndirectCause', 'RootCause']

Duplicates Removed: 0

SI_No reordered

Missing Values Summary:
Port                            1066
IncidentDetails                    6
IncidentEventDetails             123
ImmediateAction                   22
Possibility_Recurrence            23
AreaOfConcern                    183
DetailedExtentofDamageInjury     206
DirectCause                      330
Proposedcorrectiveactions         59
Corrective_Action                591
LeariningPotential               304
SeverityOfIncident               175
RevisedIncidentCategory         1369
Cause_Analysis                    23
Accident_Details                 916
LessionLearnt                    996
ReviewComments                  1126
ClosingNote                      164
ClosingDate                      205
Injured_Person_Name             1216
Injured_Person_Rank             1012
OtherRank                        841
OtherUser 

C:\Users\vinyt\AppData\Local\Temp\ipykernel_27024\243517226.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.replace(



Saved: cleaned_ADIDatabase_2015.xlsx
